### Finetune Your Own BioMedGPT

BioMedGPT is composed of a language model, a molecule encoder, a protein encoder and two modality adaptors. 

Here, we provide an example for finetuning your own model with your own recipe.

In [ ]:
# Change working directory
import os
import sys
parent = os.path.dirname(os.path.abspath(''))
print(parent)
sys.path.append(parent)
os.chdir(parent)

import logging
logging.basicConfig(level=logging.ERROR)

#### Step 1: Prepare Model Checkpoints

First of all, you need to choose the base model for your BioMedGPT.
* Large language model, e.g., DeepSeek-R1-Qwen-14B
* Molecule encoder, e.g., GraphMVP
* Protein encoder, e.g., ESM2-3B

Download and put them under `./checkpoints/`.

#### Step 2: Configurate Your BioMedGPT

Then, edit the config file `./configs/model/biomedgpt.yaml`.

```yaml
model:
  name: biomedgpt
  is_train: true
  molecule:
    # Update the ckpt path to your molecule encoder
    model_name_or_path: ./checkpoints/graphmvp/pretraining_model.pth 
    gin_hidden_dim: 300
    gin_num_layers: 5
    drop_ratio: 0
    max_n_nodes: 256
  protein:
    # Update path to your protein encoder
    model_name_or_path: ./checkpoints/esm2/3b
    use_float16: true
    seq_max_length: 1024
  llm:
    # Update path to your LLM
    model_name_or_path: ./checkpoints/biomedgpt-lm
    use_float16: true
  text_generation:
    num_beams: 2
    max_length: 512
    min_length: 1
    repetition_penalty: 1.5
    length_penalty: 1
    temperature: 0

  freeze_mol_structure_encoder: true
  freeze_prot_structure_encoder: true
  # True for Multimodal SFT
  # False for Corss-modal Alignment
  freeze_llm: false
```

#### Step 3: Prepare Your Data

Next, prepare your multimodal data. Your data should follow the format.
```json
[
    {
        "smiles": ["SMILES String", ], // can be [None]
        "sequence": ["Protein Sequence String", ], // can be [None]
        "question": "Input Query",
        "answer": "Output Answer"
    }
]
```

Here is a script example for dataset generation. You will find the `train.json` file in `./datasets/multimodal_question_answering/biomedical_qa/` after execution.

In [5]:
import os
import json

data = [
    # Molecule QA
    {
        "smiles": ["CN(C(=O)N)N=O"],
        "sequence": [None],
        "question": "Please describe this drug.",
        "answer": "<think>\nOkay, the user provided both GNN representation and SMILES notation. SMILES is a way to represent chemical structures using text strings. Now, the user is asking me to describe this drug. To do that, I need to figure out its structure, properties, uses, and perhaps its role in the body or in medical applications.\n</think>\n\nHere is a brief description of this drug. The molecule is a member of the class of N-nitrosoureas that is urea in which one of the nitrogens is substituted by methyl and nitroso groups. It has a role as a carcinogenic agent, a mutagen, a teratogenic agent and an alkylating agent."
    },
    # Protein QA
    {
        "smiles": [None],
        "sequence": ["MGCTLSAEDKAAVERSKMIDRNLREDGEKAAREVKLLLLGAGESGKSTIVKQMKIIHEAGYSEEECKQYKAVVYSNTIQSIIAIIRAMGRLKIDFGDSARADDARQLFVLAGAAEEGFMTAELAGVIKRLWKDSGVQACFNRSREYQLNDSAAYYLNDLDRIAQPNYIPTQQDVLRTRVKTTGIVETHFTFKDLHFKMFDVGGQRSERKKWIHCFEGVTAIIFCVALSDYDLVLAEDEEMNRMHESMKLFDSICNNKWFTDTSIILFLNKKDLFEEKIKKSPLTICYPEYAGSNTYEEAAAYIQCQFEDLNKRKDTKEIYTHFTCATDTKNVQFVFDAVTDVIIKNNLKDCGLF"],
        "question": "There is a mutation from E to L at position 245. What is the mutational effects?",
        "answer": "<think>\nOkay, I'm trying to figure out the mutational effects. Hmm, that looks a bit complicated, but I can try to break it down.\n\nFirst, I'm thinking about the function of the provided protein. Guanine nucleotide-binding proteins (G proteins) function as transducers downstream of G protein-coupled receptors (GPCRs) in numerous signaling cascades. The alpha chain contains the guanine nucleotide binding site and alternates between an active, GTP-bound state and an inactive, GDP-bound state. Signaling by an activated GPCR promotes GDP release and GTP binding. The alpha subunit has a low GTPase activity that converts bound GTP to GDP, thereby terminating the signal. Both GDP release and GTP hydrolysis are modulated by numerous regulatory proteins. Signaling is mediated via effector proteins, such as adenylate cyclase. Inhibits adenylate cyclase activity, leading to decreased intracellular cAMP levels (By similarity). The inactive GDP-bound form prevents the association of RGS14 with centrosomes and is required for the translocation of RGS14 from the cytoplasm to the plasma membrane. Required for normal cytokinesis during mitosis. Required for cortical dynein-dynactin complex recruitment during metaphase. \n\nThen there's a mutation from E to L at position 245. What is the mutational effects? Wait. I know it.\n</think>\n\nEnhances interaction (inactive GDP-bound) with RGS14."

    },
    # Text-only QA
    {
        "smiles": [None],
        "sequence": [None],
        "question": "A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient? (A) Ampicillin (B) Ceftriaxone (C) Doxycycline (D) Nitrofurantoin.",
        "answer": "<think>\nOkay, let's see. I have gone through the provided condition. It's a typical medical problem. Let's look at the options. (A) Ampicillin (B) Ceftriaxone (C) Doxycycline (D) Nitrofurantoin. I should choose the correct answer from A, B, C, and D.\n</think>\n<answer>\nD\n</answer>"
    }
]

path_to_data = "./datasets/multimodal_question_answering/biomedical_qa"
if not os.path.exists(path_to_data):
    os.makedirs(path_to_data, exist_ok=True)

with open(os.path.join(path_to_data, "train.json"), 'w') as f:
    json.dump(data, f, indent=4)

#### Step 4: Finetuning

Here is a script example for model finetuning.

```bash
export CUDA_VISIBLE_DEVICES=0
TASK="multimodal_question_answering"
MODEL="biomedgpt"
DATASET="biomedical_qa"

python open_biomed/scripts/train.py \
--task $TASK \
--additional_config_file configs/model/$MODEL.yaml \
--dataset_name $DATASET \
--dataset_path ./datasets/$TASK/$DATASET \
--batch_size_train 1 \
--batch_size_eval 1 \
--max_epochs 5 \
--ckpt_freq 1 \
--empty_folder 
```

The output logs and results will be saved in `./logs/multimodal_question_answering/biomedgpt-biomedical_qa/train/`.